# Phase 3: Pathogen Gate and Domain Questions

First Jev scores whether each species–abstract pair provides human disease evidence. Pairs with `p_yes` at or above `confidence_threshold` (default 0.75) then receive all 73 V1 domain questions in one Jev Decisions API request per species–abstract pair. Scores are cached and resumable.

Run the configuration cell in notebook 00 first to change Phase 3 settings. Set `RUN_PHASE_OVERRIDE = True` (or `run_phase: true` in `configs/notebook_parameters.yaml`) to run both Jev stages.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v2").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v2 import notebook_utils as nbutils
from pathogen_atlas_v2.common.io import ensure_parent, resolve_path
from pathogen_atlas_v2.llm.pathogen_evidence import apply_confidence_threshold, phase3_preflight, run_phase3
from pathogen_atlas_v2.llm.domain_expert import (
    apply_domain_question_threshold,
    domain_expert_preflight,
    load_current_domain_expert_probabilities,
    run_domain_expert_phase,
)

NOTEBOOK_CONFIG_PATH = "configs/notebook_parameters.yaml"
PARAMS = nbutils.notebook_phase_parameters("phase3_abstract_decisions", NOTEBOOK_CONFIG_PATH)
RUN_PHASE_OVERRIDE = None  # Set True to run now, False to skip, or None to use YAML.
RUN_PHASE = bool(PARAMS.get("run_phase", False)) if RUN_PHASE_OVERRIDE is None else bool(RUN_PHASE_OVERRIDE)
RUN_ID = str(PARAMS.get("run_id", "jev-human-pathogen-evidence-v1"))
DECISION_PATH = resolve_path(PARAMS.get("decision_path", "data/03_decisions/parsed/pathogen_evidence_jev.parquet"))
ACCEPTED_DECISIONS_PATH = resolve_path(PARAMS.get("accepted_decisions_path", "data/03_decisions/parsed/pathogen_evidence_jev_accepted.parquet"))
CONFIDENCE_THRESHOLD = float(PARAMS.get("confidence_threshold", 0.75))
QUESTION_CONFIDENCE_THRESHOLD = float(PARAMS.get("question_confidence_threshold", 0.75))
EXPERT_ANSWERS_PATH = resolve_path(PARAMS.get("expert_answers_path", "data/03_decisions/parsed/domain_expert_answers.parquet"))

## Preflight

This reads Phase 2 release tables and the local decision ledger. It does not call Jev. Check the pending pair count, selected model, question version, and maximum call count before enabling the run.

In [2]:
PREFLIGHT = phase3_preflight(PARAMS)
PREFLIGHT["confidence_threshold"] = CONFIDENCE_THRESHOLD
display(pd.DataFrame([PREFLIGHT]))

,status,provider,model,question_id,question_version,run_id,selected_species_pmid_pairs,species_count,missing_abstract_pairs,already_completed_for_run,pending_pairs,max_calls,scheduled_api_calls,decision_path,raw_response_path,confidence_threshold
0,ready,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,161471,5969,6,25,161446,None,161440,/Users/f.costa/Documents/Projects/Repositories...,/Users/f.costa/Documents/Projects/Repositories...,0.75


## OpenRouter credentials

Jev needs `OPENROUTER_API_KEY` in the current notebook kernel. The next cell first tries the existing macOS Keychain item for `openrouter.ai`; if it is unavailable, it asks for the key using hidden input. The key is held only in this kernel session and is not written to the notebook or YAML config.

In [3]:
import os
import subprocess
import sys
from getpass import getpass

if RUN_PHASE:
    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip()
    if not api_key and sys.platform == "darwin":
        keychain_result = subprocess.run(
            ["/usr/bin/security", "find-generic-password", "-s", "openrouter.ai", "-w"],
            capture_output=True,
            text=True,
            check=False,
        )
        if keychain_result.returncode == 0:
            api_key = keychain_result.stdout.strip()
    if not api_key:
        api_key = getpass("OpenRouter API key (input hidden): ").strip()
    if not api_key:
        raise RuntimeError("No OpenRouter API key was provided. Store it in the macOS Keychain, enter it above, or set OPENROUTER_API_KEY in this kernel.")
    os.environ["OPENROUTER_API_KEY"] = api_key
    del api_key
    print("OpenRouter key is available to this kernel session.")
else:
    print("RUN_PHASE is false; skipping OpenRouter credential setup.")

OpenRouter key is available to this kernel session.


## Run Jev decisions

The runner skips successful rows whose run ID, input hash, model, and question version match. Failed responses remain eligible for a later retry. Keep `max_calls` at 25 for a pilot; set it to `None` only when you intend to process all pending pairs.

In [4]:
if RUN_PHASE:
    if PREFLIGHT.get("status") != "ready":
        raise RuntimeError(f"Phase 3 is not ready: {PREFLIGHT}")
    PHASE3_RESULT = run_phase3(PARAMS, log=True)
else:
    PHASE3_RESULT = {
        "status": "not_run",
        "reason": "Set run_phase: true in the Phase 3 notebook parameters to call Jev.",
        "preflight": PREFLIGHT,
    }
display(PHASE3_RESULT)

Jev pathogen evidence: 161440 requests scheduled; 6 selected pairs lack abstracts; run_id=jev-human-pathogen-evidence-v1; model=typesafe/jev-1.14.
Jev pathogen evidence: 1/161440 requests complete; ETA 151h 16m 26s; estimated finish 19:23:41 CET local.; account credits remaining $6.3776; estimated ending balance waiting for OpenRouter to report spend
Jev pathogen evidence: 101/161440 requests complete; ETA 3h 27m 53s; estimated finish 15:35:13 CET local.; account credits remaining $6.3776; estimated ending balance waiting for OpenRouter to report spend
Jev pathogen evidence: 201/161440 requests complete; ETA 2h 26m 23s; estimated finish 14:33:46 CET local.; account credits remaining $6.3776; estimated ending balance waiting for OpenRouter to report spend
Jev pathogen evidence: 301/161440 requests complete; ETA 1h 51m 43s; estimated finish 13:59:07 CET local.; account credits remaining $6.3776; estimated ending balance waiting for OpenRouter to report spend
Jev pathogen evidence: 401/16

{'status': 'completed',
 'source': 'openrouter_jev',
 'model': 'typesafe/jev-1.14',
 'run_id': 'jev-human-pathogen-evidence-v1',
 'record_counts': {'decisions': 161471,
  'completed': 161465,
  'failed': 0,
  'missing_abstract': 6},
 'outputs': {'decisions': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/03_decisions/parsed/pathogen_evidence_jev.parquet',
  'raw_responses': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/03_decisions/raw_responses/jev/pathogen_evidence.jsonl'}}

## Apply the confidence threshold

The notebook labels a completed decision as accepted when `p_yes >= confidence_threshold`. The default is 0.75 and can be changed in `configs/notebook_parameters.yaml`. Failed or missing decisions remain unaccepted. The raw probability ledger is unchanged; accepted rows are written to a separate Parquet file.


In [5]:
if DECISION_PATH.exists():
    DECISIONS = pd.read_parquet(DECISION_PATH)
    if "run_id" in DECISIONS.columns:
        CURRENT_RUN = DECISIONS.loc[DECISIONS["run_id"].astype(str).eq(RUN_ID)].copy()
    else:
        CURRENT_RUN = DECISIONS.iloc[0:0].copy()
    CURRENT_RUN = apply_confidence_threshold(CURRENT_RUN, CONFIDENCE_THRESHOLD)
    ACCEPTED_DECISIONS = CURRENT_RUN.loc[CURRENT_RUN["accepted"].fillna(False)].copy()
    ensure_parent(ACCEPTED_DECISIONS_PATH)
    ACCEPTED_DECISIONS.to_parquet(ACCEPTED_DECISIONS_PATH, index=False)
    if CURRENT_RUN.empty:
        print(f"No saved decisions for run {RUN_ID!r} yet.")
    else:
        display(CURRENT_RUN["status"].value_counts(dropna=False).rename_axis("status").to_frame("rows"))
        display(CURRENT_RUN["accepted"].value_counts(dropna=False).rename_axis("accepted at threshold").to_frame("rows"))
        display(ACCEPTED_DECISIONS.head(20))
        print(f"Accepted {len(ACCEPTED_DECISIONS)} of {len(CURRENT_RUN)} decisions at p_yes >= {CONFIDENCE_THRESHOLD:.2f}.")
    print(f"Accepted decisions saved to: {ACCEPTED_DECISIONS_PATH}")
else:
    print(f"Decision ledger does not exist yet: {DECISION_PATH}")


,rows
status,
completed,161465
missing_abstract,6


,rows
accepted at threshold,
False,134434
True,27031
<NA>,6


,pmid,canonical_species,matched_names,input_hash,provider,model,question_id,question_version,run_id,p_yes,status,attempts,error,completed_at,raw_response_path,confidence_threshold,accepted
2,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",1b60e520b975d5639330f0960386f65dca8d4c355b11ab...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.98,completed,1,None,2026-10-30T08:32:50+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True
3,10203522,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",c7824fb68117ec01d2c45c76f65a901f6bbc4022313a0c...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.99,completed,1,None,2026-10-30T08:32:50+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True
5,10629690,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",67100116f05e55f43ab067a30a568daf2291a02ad69a1c...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.98,completed,1,None,2026-10-30T08:32:51+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True
7,10681362,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",d346de61926a80cf30fa83ef44027420ee9f269999bbea...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.99,completed,1,None,2026-10-30T08:32:51+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True
9,11256786,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",f4b18d3c75a82b8cb68fc98a6d98a935b91f24d040e190...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.88,completed,1,None,2026-10-30T08:32:51+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True
10,11095068,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",71d7dc13a43938e0816ddf7cd545e617cf772f6943ca2c...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.98,completed,1,None,2026-10-30T08:32:51+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True
12,12237606,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",41345731750f767d8132830e00e4da789c2ba99e9663a9...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,1.00,completed,1,None,2026-10-30T08:32:51+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True
13,12435072,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",5fc77dfdefbe1a5a2c52bbfc1eb5b56507e611f60015d5...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.99,completed,1,None,2026-10-30T08:32:51+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True
14,12543919,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",0ef9f8b916de00323cbce45ee48470c1104437b89b61dc...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.98,completed,1,None,2026-10-30T08:32:52+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True
16,15194844,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",d777310f7b5131dd4dd37af16cbfbe120b2351a657ed8e...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.99,completed,1,None,2026-10-30T08:32:52+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True


Accepted 27031 of 161471 decisions at p_yes >= 0.75.
Accepted decisions saved to: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/03_decisions/parsed/pathogen_evidence_jev_accepted.parquet


## Domain expert questions

The saved pathogen-gate ledger is applied first. Only unchanged species–abstract pairs from the current gate run with completed `p_yes >= confidence_threshold` enter this stage. Jev receives the title and abstract once and answers all 73 versioned V1 yes/no questions together in one typed request.

Each species–abstract request is cached as raw JSONL; normalized question probabilities are checkpointed to Parquet. `expert_max_calls` limits species–abstract requests per run, not individual questions.


In [6]:
EXPERT_PREFLIGHT = domain_expert_preflight(PARAMS)
display(pd.DataFrame([EXPERT_PREFLIGHT]))


,status,provider,model,question_set_id,question_set_version,questions_per_abstract,gate_run_id,gate_threshold,gate_passing_species_abstract_pairs,already_completed_pairs,pending_pairs,max_calls,scheduled_api_calls,answers_scheduled,decision_path
0,ready,openrouter_jev,typesafe/jev-1.14,domain_expert_features,v1-domain-expert-questions-73-v1,73,jev-human-pathogen-evidence-v1,0.75,27031,17,27014,None,27014,1972022,/Users/f.costa/Documents/Projects/Repositories...


## Run and inspect domain questions

When `RUN_PHASE` is enabled, this runs the expert-question stage after the pathogen gate. Each returned probability is converted to a yes/no answer using `question_confidence_threshold` (default 0.75); failed questions remain blank.


In [10]:
if RUN_PHASE:
    if EXPERT_PREFLIGHT.get("status") != "ready":
        raise RuntimeError(f"Domain expert questions are not ready: {EXPERT_PREFLIGHT}")
    EXPERT_RESULT = run_domain_expert_phase(PARAMS, log=True)
else:
    EXPERT_RESULT = {"status": "not_run", "preflight": EXPERT_PREFLIGHT}
display(EXPERT_RESULT)

EXPERT_PROBABILITIES = load_current_domain_expert_probabilities(PARAMS)
EXPERT_ANSWERS = apply_domain_question_threshold(
    EXPERT_PROBABILITIES,
    QUESTION_CONFIDENCE_THRESHOLD,
)
ensure_parent(EXPERT_ANSWERS_PATH)
EXPERT_ANSWERS.to_parquet(EXPERT_ANSWERS_PATH, index=False)
if EXPERT_ANSWERS.empty:
    print("No saved domain-question answers for the current gate-passing abstracts yet.")
else:
    display(EXPERT_ANSWERS.groupby(["category", "answer_yes"], dropna=False).size().rename("rows").to_frame())
    display(EXPERT_ANSWERS.head(40))
print(f"Domain-question answers saved to: {EXPERT_ANSWERS_PATH}")


Jev domain questions: 73 questions per abstract; 27031 gate-passing pairs; 2532 uncached; 2532 requests scheduled; model=typesafe/jev-1.14.
Jev domain questions: 1/2532 abstract requests complete; 73/184836 question outcomes recorded. ETA 4h 53m 15s; estimated finish 16:06:44 GMT local.; account credits remaining $7.0854; estimated ending balance waiting for OpenRouter to report spend
Jev domain questions: 101/2532 abstract requests complete; 7373/184836 question outcomes recorded. ETA 12m 22s; estimated finish 11:26:15 GMT local.; account credits remaining $6.9945; estimated ending balance $4.8060
Jev domain questions: 201/2532 abstract requests complete; 14673/184836 question outcomes recorded. ETA 8m 22s; estimated finish 11:22:27 GMT local.; account credits remaining $6.9945; estimated ending balance $4.8960
Jev domain questions: 301/2532 abstract requests complete; 21973/184836 question outcomes recorded. ETA 7m 26s; estimated finish 11:21:48 GMT local.; account credits remaining 

{'status': 'completed',
 'provider': 'openrouter_jev',
 'model': 'typesafe/jev-1.14',
 'gate_threshold': 0.75,
 'question_count': 73,
 'record_counts': {'question_answer_rows': 1973263,
  'passing_pairs': 27031,
  'completed': 1973263,
  'failed': 0},
 'outputs': {'probabilities': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/03_decisions/parsed/domain_expert_probabilities.parquet',
  'raw_responses': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/03_decisions/raw_responses/jev/domain_expert_questions.jsonl'}}

rows
category                answer_yes        
animal_reservoir        False       212690
                        True          3558
bacterial_physiology    False       158143
                        True          4043
clinical_presentation   False       314263
                        True         37140
environmental_reservoir False       214763
                        True          1485
healthcare_associated   False       106927
                        True          1197
infection_context       False       123794
                        True         11361
other                   False       129497
                        True          5658
patient                 False       223611
                        True         19668
public_health           False        78303
                        True          2790
transmission            False       320001
                        True          4371

,pmid,canonical_species,matched_names,input_hash,gate_run_id,gate_p_yes,provider,model,question_set_id,question_set_version,...,concept,category,p_yes,status,attempts,error,completed_at,raw_response_path,question_confidence_threshold,answer_yes
0,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",1b60e520b975d5639330f0960386f65dca8d4c355b11ab...,jev-human-pathogen-evidence-v1,0.98,openrouter_jev,typesafe/jev-1.14,domain_expert_features,v1-domain-expert-questions-73-v1,...,AIR,environmental_reservoir,0.02,completed,1,None,2026-10-30T08:33:06.250277+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,False
1,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",1b60e520b975d5639330f0960386f65dca8d4c355b11ab...,jev-human-pathogen-evidence-v1,0.98,openrouter_jev,typesafe/jev-1.14,domain_expert_features,v1-domain-expert-questions-73-v1,...,AIRBORNE,transmission,0.01,completed,1,None,2026-10-30T08:33:06.250277+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,False
2,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",1b60e520b975d5639330f0960386f65dca8d4c355b11ab...,jev-human-pathogen-evidence-v1,0.98,openrouter_jev,typesafe/jev-1.14,domain_expert_features,v1-domain-expert-questions-73-v1,...,ALCOHOL,patient,0.01,completed,1,None,2026-10-30T08:33:06.250277+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,False
3,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",1b60e520b975d5639330f0960386f65dca8d4c355b11ab...,jev-human-pathogen-evidence-v1,0.98,openrouter_jev,typesafe/jev-1.14,domain_expert_features,v1-domain-expert-questions-73-v1,...,ANAEROBIC,bacterial_physiology,0.04,completed,1,None,2026-10-30T08:33:06.250277+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,False
4,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",1b60e520b975d5639330f0960386f65dca8d4c355b11ab...,jev-human-pathogen-evidence-v1,0.98,openrouter_jev,typesafe/jev-1.14,domain_expert_features,v1-domain-expert-questions-73-v1,...,ANIMAL CONTACT,animal_reservoir,0.01,completed,1,None,2026-10-30T08:33:06.250277+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,False
5,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",1b60e520b975d5639330f0960386f65dca8d4c355b11ab...,jev-human-pathogen-evidence-v1,0.98,openrouter_jev,typesafe/jev-1.14,domain_expert_features,v1-domain-expert-questions-73-v1,...,AQUATIC VERTEBRATES,animal_reservoir,0.01,completed,1,None,2026-10-30T08:33:06.250277+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,False
6,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",1b60e520b975d5639330f0960386f65dca8d4c355b11ab...,jev-human-pathogen-evidence-v1,0.98,openrouter_jev,typesafe/jev-1.14,domain_expert_features,v1-domain-expert-questions-73-v1,...,ASYMPTOMATIC CARRIAGE,bacterial_physiology,0.02,completed,1,None,2026-10-30T08:33:06.250277+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,False
7,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",1b60e520b975d5639330f0960386f65dca8d4c355b11ab...,jev-human-pathogen-evidence-v1,0.98,openrouter_jev,typesafe/jev-1.14,domain_expert_features,v1-domain-expert-questions-73-v1,...,BACTERAEMIA,clinical_presentation,0.03,completed,1,None,2026-10-30T08:33:06.250277+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,False
8,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",1b60e520b975d5639330f0960386f65dca8d4c355b11ab...,jev-human-pathogen-evidence-v1,0.98,openrouter_jev,typesafe/jev-1.14,domain_expert_features,v1-domain-expert-questions-73-v1,...,BIOFILM,bacterial_physiology,0.02,completed,1,None,2026-10-30T08:33:06.250277+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,False
9,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Strep

Domain-question answers saved to: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/03_decisions/parsed/domain_expert_answers.parquet
